# Puckworks — Full Laboratory Tour (browser, no terminal)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trbrewer/puckworks/blob/main/notebooks/guided_pull_laboratory_colab.ipynb)

Tour the models and scientific components used throughout Puckworks — in your browser, no install, no terminal, no code to type. Fill in the short **form**, then press the single **▶ Run the Laboratory** button.

This is an honest research tool, **not** a digital twin, recipe optimizer, or taste predictor. Different components answer different questions: some use *your* recipe, some run their *own* reference case, and some run a scientific *check* — these are **not** directly comparable, and are never averaged or ranked. It runs **privately** in your Colab runtime; that does not make any model cleared for public hosting.

**Development preview (0.4.0.dev0)** — not the v0.3.0 public release.

## 1. Set up (runs automatically — nothing to type)

In [ ]:
# This runs automatically — you do NOT type anything here.
# DEVELOPMENT PREVIEW: the Guided Pull Laboratory is 0.4.0.dev0 (NOT the v0.3.0 public release).
# It installs an EXACT, immutable pinned commit of the source (no mutable "latest main").
import os, subprocess, sys, hashlib

PIN_COMMIT = "a366285853cc208d511b31a8a1c9a00986a582bf"
INSTALL_SOURCE = f"git+https://github.com/trbrewer/puckworks@{PIN_COMMIT}"

override = os.environ.get("PUCKWORKS_WHEEL", "").strip()
if override:
    if not os.path.isfile(override):
        raise SystemExit("PUCKWORKS_WHEEL is set but is not a file: " + override)
    print("Hermetic mode — installing local candidate wheel")
    print("  wheel sha256:", hashlib.sha256(open(override, "rb").read()).hexdigest())
    INSTALL_SOURCE = override
else:
    print("DEVELOPMENT PREVIEW — installing pinned commit", PIN_COMMIT)

# Every dev build shares the version 0.4.0.dev0, so a plain `pip install` would SKIP an already-present
# copy (leaving stale code). Uninstall first, then install the pinned build, so the pinned code is really
# what runs.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "puckworks"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", INSTALL_SOURCE], check=True)
print("Install complete. install_source =", INSTALL_SOURCE)

# If puckworks was already imported earlier in THIS runtime, Python keeps the OLD modules cached and the
# fresh install cannot take effect until the runtime restarts. Detect that and stop with clear guidance.
if "puckworks" in sys.modules:
    raise SystemExit(
        "An older puckworks was already loaded in this Colab session. The new install cannot replace "
        "already-imported modules. Fix: menu Runtime -> Restart runtime (or Disconnect and delete "
        "runtime), then Runtime -> Run all.")

## 2. Environment check

In [ ]:
import puckworks
print("puckworks version :", puckworks.__version__)
print("pinned commit     :", PIN_COMMIT)
print("install source    :", INSTALL_SOURCE)
print("components         :", len(puckworks.components()))
# DEVELOPMENT PREVIEW (0.4.0.dev0) — do not read it as the v0.3.0 public release.
assert puckworks.__version__ == "0.4.0.dev0", puckworks.__version__
# Fail fast with clear guidance if the installed build is stale (missing the tour modules): this happens
# when an older puckworks was already loaded in this Colab session.
try:
    from puckworks.product import lab_tour  # noqa: F401
except ImportError:
    raise SystemExit(
        "This runtime still has an OLDER puckworks without the Full Laboratory Tour. Fix: menu "
        "Runtime -> Restart runtime (or Disconnect and delete runtime), then Runtime -> Run all.")
print("Full Laboratory Tour available: yes")

## 3. Choose your tour

`Full Laboratory Tour` (default) resolves every component; `Quick Tour` runs the fast subset; `Your Shot Only` runs just the recipe models; `Model Library Only` browses and runs nothing. A grinder dial is **not** a universal particle size, so there is no grind→size control here.

In [ ]:
#@title Your tour { display-mode: "form", run: "auto" }
#@markdown Choose what to do, then run the **Run the Laboratory** cell below.
experience_mode = "Full Laboratory Tour" #@param ["Full Laboratory Tour", "Quick Tour", "Your Shot Only", "Model Library Only"]
preset = "pv19_named" #@param ["pv19_named", "guided_v1"]
dose_g = 20.0 #@param {type:"number"}
target_beverage_g = 40.0 #@param {type:"number"}
pressure_bar = 9.0 #@param {type:"number"}

### Advanced (optional)

In [ ]:
#@title Advanced (optional) { display-mode: "form", run: "auto" }
#@markdown Most people can leave these as-is.
brew_temperature_c = 93.0 #@param {type:"number"}
outside_evidence_range = "Show the result with a warning" #@param ["Show the result with a warning", "Do not run outside the evidence range"]

## 4. Coverage preview (before you run)

In [ ]:
# Coverage preview (before you run) — how many components will actually execute, and why not all of them.
from puckworks.product import lab_tour
from collections import Counter
_plans = lab_tour.tour_manifest()
_kinds = Counter(p.execution_kind.value for p in _plans.values())
print("The Full Laboratory Tour resolves ALL", len(_plans), "registered components.")
print("  will run the entered recipe (common scenario):", _kinds.get("COMMON_SCENARIO", 0))
print("  will run a native reference case            :", _kinds.get("NATIVE_REFERENCE", 0))
print("  will run a registered scientific check      :", _kinds.get("SCIENTIFIC_CHECK", 0))
print("  optional dependency unavailable             :", _kinds.get("OPTIONAL_DEPENDENCY", 0))
print("  rights-blocked (shown, never run)           :", _kinds.get("RIGHTS_BLOCKED", 0))
print()
print("Not all results use your entered shot: native cases run the model's OWN reference case, and")
print("scientific checks run a model's registered gate on its own data. These are NOT predictions of")
print("your shot and are NOT directly comparable to each other.")

## 5. Run the Laboratory

Press ▶ on the cell below. It runs your chosen tour **privately** (`LOCAL_PRIVATE`).

In [ ]:
#@title ▶ Run the Laboratory { display-mode: "form" }
# Press the play button on THIS cell. Everything runs PRIVATELY in this Colab runtime (context
# LOCAL_PRIVATE). "Private" means it runs in a runtime you control; it does NOT mean any model is
# cleared for public hosting.
from puckworks.product import lab, lab_service, lab_explorer, lab_tour

_overrides = {"dose_g": float(dose_g), "target_beverage_g": float(target_beverage_g),
              "pressure_bar": float(pressure_bar), "brew_temperature_c": float(brew_temperature_c)}
_domain_policy = "strict" if outside_evidence_range.startswith("Do not run") else "warn"
mode = experience_mode
tour = None; lab_result = None; catalog = None

if mode == "Model Library Only":
    catalog = lab_explorer.explorer_catalog()
    print("Model library:", catalog["n_components"], "models; no scientific producer ran.")
elif mode == "Full Laboratory Tour":
    req = lab.ScenarioRequest(preset, overrides=_overrides, domain_policy=_domain_policy)
    print("Running the Full Laboratory Tour across all espresso stages... (about 20-30 s)")
    tour = lab_tour.execute_laboratory_tour(req, execution_context="LOCAL_PRIVATE")
    s = tour["summary"] if isinstance(tour, dict) else tour.summary
    print("Resolved", s["registered"], "components;", s["completed"], "executed;",
          s["rights_blocked"], "rights-blocked;", s["optional_unavailable"], "optional-unavailable.")
elif mode == "Your Shot Only":
    req = lab.ScenarioRequest(preset, overrides=_overrides, domain_policy=_domain_policy,
                              lens_selection_policy="all_ready", reference_selection_policy="none")
    lab_result = lab_service.execute_lab_request(req, execution_context="LOCAL_PRIVATE")
else:  # Quick Tour
    req = lab.ScenarioRequest(preset, overrides=_overrides, domain_policy=_domain_policy,
                              lens_selection_policy="primary", reference_selection_policy="interactive_fast")
    lab_result = lab_service.execute_lab_request(req, execution_context="LOCAL_PRIVATE")
print("Done. Scroll down for results grouped by stage.")

In [ ]:
# Plain-language badge for each component result (never lead with the internal component id).
_BADGE = {
    "COMMON_SCENARIO": "USES YOUR SHOT",
    "NATIVE_REFERENCE": "NATIVE REFERENCE CASE",
    "SCIENTIFIC_CHECK": "SCIENTIFIC CHECK",
    "OPTIONAL_DEPENDENCY": "OPTIONAL DEPENDENCY UNAVAILABLE",
    "RIGHTS_BLOCKED": "RIGHTS BLOCKED",
    "NO_EXECUTION_PATH": "NO EXECUTION PATH",
}
def _badge(comp):
    if comp["execution_status"] == "EXECUTION_ERROR":
        return "EXECUTION ERROR"
    if comp["execution_status"] in ("RIGHTS_NOT_CLEARED",):
        return "NOT CLEARED HERE"
    return _BADGE.get(comp["execution_kind"], comp["execution_kind"])

## 6. Results by stage (quick overview)

In [ ]:
# Results organized into plain-language sections. Each card leads with a badge, not an id.
if catalog is not None:
    print("Model library (no execution):", catalog["n_components"], "models.")
    print("Publicly runnable today:", ", ".join(catalog["public_live_component_ids"]) or "(none yet)")
elif tour is not None:
    from puckworks.product import lab_tour_display as _disp
    for _name, _cards in _disp.tour_display_sections(tour):
        print("")
        print("##### " + _name + " #####")
        for c in _cards:
            if "technical" in c:
                print("  [" + c["badge"] + "]  " + c["headline"])
                print("      what ran: " + c["what_ran"])
                print("      inputs: " + c["inputs"])
                for o in (c["outputs"] or [])[:2]:
                    if "value" in o:
                        print("        - " + o["name"] + ": " + str(o["value"]) + " " + o.get("unit","") + " [" + o.get("role","") + "]")
                    elif "gate_id" in o:
                        print("        - gate " + o["gate_id"] + ": " + o["status"])
                print("      comparable? " + c["comparable"])
                print("      does NOT establish: " + c["does_not_establish"])
                print("      (technical id: " + c["technical"]["component_id"] + ")")
            else:
                print("  " + c.get("headline","") + " - " + (c.get("what_ran") or c.get("manifest_id","")))
elif lab_result is not None and not lab_result.blocked:
    rep = lab_result.report
    for lens in rep.get("executed_lenses", []):
        print("Model that used your recipe:", lens["component_id"])
        for o in lens["observables"][:4]:
            print("  " + o["name"] + ": " + str(o["value"]) + " " + o["unit"] + " [" + o["role"] + "]")
    for r in rep.get("executed_reference_results", []):
        print("Native reference:", r["component_id"], "-", r["status"])
    for sline in rep["what_this_does_not_prove"][:3]:
        print("-", sline)
else:
    print("The request was blocked or produced no result.")

## 7. Per-model deep dive

The deep dive **begins with the one model that turns your whole recipe into a simulated shot** — Cameron (2020) — and looks at it closely: the whole shot over time, then how the model responds when we sweep the pressure and the beverage mass. Then we take the shot apart and visit each specialist station, in the order the espresso process happens.

Each model reads like one small exhibit: a public heading, a one-line *what it computes*, then **zero, one, or several** figures — no filler. Every figure teaches ONE idea, laid out with a reserved evidence header (badge) and footer (provenance + a `Scope:` caveat) that never sit on the plot. Below each figure the explanation is split into short labelled parts — *What changes*, *What the model shows*, *Why this happens*, and a *Scope* note — and all of the technical evidence (badge, evidence strength, varied and held-fixed inputs, the full fidelity ceiling, the reference, and the raw identifiers) is tucked into a collapsible **Evidence and technical details** section so you can reach it without wading through it. Numbers in every explanation are recomputed from the model, never typed by hand; scientific-check (gate) numbers are technical evidence, **not** novice figures, so they are not plotted here.

In [ ]:
# Per-model deep dive (Full Laboratory Tour). Cameron is the HERO: the one model that turns the whole
# recipe into a simulated shot, shown first with several figures. Then we take the shot apart and visit
# each specialist station in espresso process order. Figures come from lab_tour_insights.component_figures
# (0/1/many per model); the calm, museum-style layout (reserved evidence header/footer, structured
# explanation, collapsed technical details) is built by lab_tour_notebook_display. Gate numbers are NOT
# plotted here.
if tour is not None:
    import puckworks
    try:  # crisp inline figures in Colab; harmless outside IPython
        get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")
    except Exception:
        pass
    try:
        get_ipython().run_line_magic("matplotlib", "inline")
    except Exception:
        pass
    from IPython.display import display, Markdown
    from puckworks.product import lab_component_stories as _st, lab_tour_insights as _ins
    from puckworks.product import lab_tour_notebook_display as _nd
    import matplotlib.pyplot as _plt
    _t = tour.to_dict() if not isinstance(tour, dict) else tour
    _byid = {c["component_id"]: c for c in _t["components"]}
    _refs = {c.name: (getattr(c, "paper", "") + (" · doi:" + c.doi if getattr(c, "doi", "") else ""))
             for c in puckworks.components()}
    # the scenario the educational producers vary AROUND (fixed conditions come from your form entries)
    _scenario = {"preset_id": preset, "overrides": _overrides, "domain_policy": _domain_policy}

    display(Markdown("> _" + _st.STANDING_DISCLAIMER + "_"))
    for _cid in _st.ordered_component_ids():
        _s = _st.component_story(_cid); _c = _byid.get(_cid)
        for _b in _nd.model_intro_blocks(_s, _cid):
            display(Markdown(_b))

        # 0, 1, or several EDUCATIONAL figures — each obeys this component's tour rights decision.
        _figs = _ins.component_figures(_c, scenario=_scenario,
                                       execution_context="LOCAL_PRIVATE") if _c else []
        if _figs:
            for _f in _figs:
                display(Markdown(_nd.figure_headline_block(_f.headline, _f.question)))
                display(_f.figure); _plt.close(_f.figure)
                for _b in _nd.narrative_blocks(_f.narrative):
                    display(Markdown(_b))
        else:
            _reason = _ins.no_figure_reason(_c) if _c else "NO_DEFENSIBLE_PUBLIC_RELATIONSHIP_YET"
            display(Markdown(_nd.no_figure_block(_reason, (_c or {}).get("message", ""))))

        display(Markdown(_nd.cup_block(_s)))
        display(Markdown(_nd.evidence_details_block(_cid, _s, _c, _refs.get(_cid, ""), _figs)))

        if _cid in _st.HERO_COMPONENT_IDS:  # transition from the whole-shot hero to the stations
            display(Markdown("> **Now that we have seen the whole simulated shot, let's take it apart and "
                             "visit the specialist stations** — each model below answers one narrower "
                             "question about a single stage of the process."))
        display(Markdown(_nd.model_separator()))
else:
    print("(The per-model deep dive appears for the Full Laboratory Tour. Pick that mode and run again.)")

## 8. Public vs private

In [ ]:
# Public vs private, in plain language.
from puckworks.product import lab_explorer
live = lab_explorer.public_live_component_ids()
print("This notebook runs PRIVATELY in your Colab runtime (context: LOCAL_PRIVATE).")
print("Private inspection can run models that are not yet cleared for PUBLIC hosting.")
print("Publicly-runnable models today (affirmative rights only):", ", ".join(live) if live else "(none)")
print("grudeva2025.reduced is rights-blocked and never runs here (issue #73).")

## 9. Downloads and technical details

In [ ]:
# Download the complete, provenance-bearing report.
import json
_payload = None
if tour is not None:
    _payload = tour.to_dict() if not isinstance(tour, dict) else tour
elif lab_result is not None and not lab_result.blocked:
    from puckworks.product import lab
    _payload = json.loads(lab.artifact_json(lab_result.report))
if _payload is not None:
    open("laboratory_tour.json", "w").write(json.dumps(_payload, indent=2, sort_keys=True, default=str))
    open("laboratory_tour.md", "w").write("# Puckworks Laboratory report\n\n```json\n"
                                          + json.dumps(_payload, indent=2, sort_keys=True, default=str)[:4000]
                                          + "\n```\n")
    print("Wrote laboratory_tour.json and laboratory_tour.md")
    try:
        from google.colab import files as _f
        _f.download("laboratory_tour.json"); _f.download("laboratory_tour.md")
    except Exception:
        print("(not in Colab; files written to the working directory)")
else:
    print("(nothing to download for Model Library Only)")

In [ ]:
# completion sentinel (used by CI to confirm the notebook ran end-to-end)
_ver = __import__("puckworks").__version__
if tour is not None:
    t = tour.to_dict() if not isinstance(tour, dict) else tour
    _mode, _n = "full_tour", "%d/%d executed" % (t["summary"]["completed"], t["summary"]["registered"])
    _hash = t["tour_scientific_hash"]
    _components = ",".join(sorted(c["component_id"] for c in t["components"]
                                  if c["execution_status"] == "EXECUTED"))
elif catalog is not None:
    _mode, _n, _hash, _components = "catalog_only", "%d models" % catalog["n_components"], "NA", "-"
elif lab_result is not None and not lab_result.blocked:
    _mode, _n = "shot", "1"
    _hash = lab_result.report["integrity"]["scientific_payload_sha256"]
    _components = ",".join(lab_result.selected_component_ids) or "lens"
else:
    _mode, _n, _hash, _components = "blocked", "0", "NA", "-"
print(f"GUIDED_PULL_LAB_COMPLETE: {_ver} · mode={_mode} · context=LOCAL_PRIVATE · "
      f"executed={_n} · manifest={lab_tour.FULL_LABORATORY_TOUR_V1} · sci_hash={_hash}")
print("  components=", _components[:200])